# Term Deposit Subscription Prediction (Bank Marketing)

## Problem Statement and Objective

A Portuguese bank ran a series of phone-based marketing campaigns to sell term deposit products. The goal of this project is to **predict whether a customer will subscribe to a term deposit** (`yes` / `no`) based on demographic, financial, and campaign-related features.

### Key Objectives
- Understand and explore the Bank Marketing dataset
- Clean and encode all features appropriately
- Train and evaluate classification models (Logistic Regression & Random Forest)
- Interpret individual predictions using SHAP (Explainable AI)

### Dataset
**Source:** UCI Machine Learning Repository — Bank Marketing Dataset  
**Target variable:** `y` — whether the client subscribed to a term deposit (`yes` = 1, `no` = 0)

## 1. Setup & Imports

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import shap
import warnings
warnings.filterwarnings('ignore')

from ucimlrepo import fetch_ucirepo
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    confusion_matrix, classification_report,
    f1_score, roc_curve, auc, ConfusionMatrixDisplay
)

sns.set_theme(style='whitegrid', palette='Set2')
plt.rcParams['figure.dpi'] = 100

# All charts/plots will be saved here
CHARTS_DIR = 'charts'
os.makedirs(CHARTS_DIR, exist_ok=True)

print('All libraries loaded successfully.')
print(f'Charts will be saved to: {os.path.abspath(CHARTS_DIR)}')

## 2. Dataset Loading & Description

In [ ]:
# Fetch dataset from UCI ML Repository (ID = 222)
bank_marketing = fetch_ucirepo(id=222)

X_raw = bank_marketing.data.features
y_raw = bank_marketing.data.targets

df = pd.concat([X_raw, y_raw], axis=1)
df.rename(columns={'y': 'subscribed'}, inplace=True)

print(f'Dataset shape: {df.shape}')
print(f'\nFeatures: {list(X_raw.columns)}')
print(f'\nTarget: subscribed (yes/no)')

In [ ]:
df.head()

In [ ]:
df.info()

In [ ]:
df.describe()

### Feature Descriptions

| Feature | Description |
|---------|-------------|
| `age` | Client age |
| `job` | Type of job |
| `marital` | Marital status |
| `education` | Education level |
| `default` | Has credit in default? |
| `balance` | Average yearly balance (euros) |
| `housing` | Has housing loan? |
| `loan` | Has personal loan? |
| `contact` | Contact communication type |
| `day` | Last contact day of the month |
| `month` | Last contact month |
| `duration` | Last contact duration (seconds) |
| `campaign` | Number of contacts during this campaign |
| `pdays` | Days since last contact (-1 = not contacted) |
| `previous` | Contacts before this campaign |
| `poutcome` | Outcome of previous campaign |
| `subscribed` | **Target** — subscribed to term deposit? |

## 3. Data Cleaning & Preprocessing

In [ ]:
# Check for missing values
print('Missing values per column:')
print(df.isnull().sum())
print(f'\nTotal missing: {df.isnull().sum().sum()}')

In [ ]:
# Check for duplicates
print(f'Duplicate rows: {df.duplicated().sum()}')
df.drop_duplicates(inplace=True)
print(f'Shape after removing duplicates: {df.shape}')

In [ ]:
# Replace 'unknown' with NaN, then fill with mode for categorical columns
categorical_cols = df.select_dtypes(include='object').columns.tolist()
categorical_cols.remove('subscribed')

for col in categorical_cols:
    df[col] = df[col].replace('unknown', np.nan)
    df[col].fillna(df[col].mode()[0], inplace=True)

print('Unknown values replaced and filled with mode.')
print(f'Remaining nulls: {df.isnull().sum().sum()}')

In [ ]:
# pdays == -1 means client was not previously contacted — replace with 0 indicator
df['was_contacted_before'] = (df['pdays'] != -1).astype(int)
df['pdays'] = df['pdays'].replace(-1, 0)

print('Feature engineering: added was_contacted_before indicator.')

### Encoding Categorical Features

In [ ]:
# Binary columns — label encode directly
binary_cols = ['default', 'housing', 'loan']
le = LabelEncoder()
for col in binary_cols:
    df[col] = le.fit_transform(df[col])

# Encode target
df['subscribed'] = df['subscribed'].map({'yes': 1, 'no': 0})

# One-hot encode remaining nominal categorical columns
nominal_cols = ['job', 'marital', 'education', 'contact', 'month', 'poutcome']
df = pd.get_dummies(df, columns=nominal_cols, drop_first=True)

print(f'Shape after encoding: {df.shape}')
df.head(2)

## 4. Exploratory Data Analysis (EDA)

In [ ]:
# Target class distribution
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

counts = df['subscribed'].value_counts()
axes[0].bar(['No (0)', 'Yes (1)'], counts.values, color=['#E74C3C', '#2ECC71'], edgecolor='black')
axes[0].set_title('Target Class Distribution', fontsize=13)
axes[0].set_ylabel('Count')
for i, v in enumerate(counts.values):
    axes[0].text(i, v + 100, str(v), ha='center', fontweight='bold')

axes[1].pie(counts.values, labels=['No', 'Yes'], autopct='%1.1f%%',
            colors=['#E74C3C', '#2ECC71'], startangle=90)
axes[1].set_title('Subscription Rate', fontsize=13)

plt.suptitle('Class Imbalance Overview', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(CHARTS_DIR, 'class_distribution.png'), bbox_inches='tight')
plt.show()
print(f'Subscription rate: {counts[1]/len(df)*100:.1f}%')

In [ ]:
# Distribution of key numerical features by target
num_features = ['age', 'balance', 'duration', 'campaign', 'previous']

fig, axes = plt.subplots(1, len(num_features), figsize=(20, 4))
for ax, col in zip(axes, num_features):
    df[df['subscribed'] == 0][col].hist(ax=ax, alpha=0.6, label='No', bins=30, color='#E74C3C')
    df[df['subscribed'] == 1][col].hist(ax=ax, alpha=0.6, label='Yes', bins=30, color='#2ECC71')
    ax.set_title(col)
    ax.legend()

plt.suptitle('Numerical Feature Distributions by Subscription', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(CHARTS_DIR, 'feature_distributions.png'), bbox_inches='tight')
plt.show()

In [ ]:
# Correlation heatmap of numerical features
num_df = df[num_features + ['subscribed']]
plt.figure(figsize=(8, 6))
sns.heatmap(num_df.corr(), annot=True, fmt='.2f', cmap='coolwarm', linewidths=0.5)
plt.title('Correlation Matrix — Numerical Features', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(CHARTS_DIR, 'correlation_heatmap.png'), bbox_inches='tight')
plt.show()

In [ ]:
# Call duration vs subscription (most predictive feature)
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

sns.boxplot(x='subscribed', y='duration', data=df,
            palette={0: '#E74C3C', 1: '#2ECC71'}, ax=axes[0])
axes[0].set_xticklabels(['No', 'Yes'])
axes[0].set_title('Call Duration by Subscription', fontsize=12)
axes[0].set_xlabel('Subscribed')

sns.boxplot(x='subscribed', y='age', data=df,
            palette={0: '#E74C3C', 1: '#2ECC71'}, ax=axes[1])
axes[1].set_xticklabels(['No', 'Yes'])
axes[1].set_title('Age by Subscription', fontsize=12)
axes[1].set_xlabel('Subscribed')

plt.suptitle('Key Feature vs Target', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(CHARTS_DIR, 'boxplots.png'), bbox_inches='tight')
plt.show()

In [ ]:
# Balance distribution (log-scale for skew)
plt.figure(figsize=(8, 4))
df_pos = df[df['balance'] > 0]
df_pos[df_pos['subscribed'] == 0]['balance'].apply(np.log1p).hist(alpha=0.6, label='No', bins=40, color='#E74C3C')
df_pos[df_pos['subscribed'] == 1]['balance'].apply(np.log1p).hist(alpha=0.6, label='Yes', bins=40, color='#2ECC71')
plt.title('Log(Balance + 1) Distribution by Subscription', fontsize=12)
plt.xlabel('log(balance + 1)')
plt.legend()
plt.tight_layout()
plt.savefig(os.path.join(CHARTS_DIR, 'balance_distribution.png'), bbox_inches='tight')
plt.show()

## 5. Model Building & Evaluation

In [ ]:
# Train/test split
X = df.drop('subscribed', axis=1)
y = df['subscribed']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f'Training set: {X_train.shape[0]} samples')
print(f'Test set:     {X_test.shape[0]} samples')
print(f'Class balance in train: {y_train.value_counts().to_dict()}')

In [ ]:
# Scale numerical features for Logistic Regression
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

### 5.1 Logistic Regression

In [ ]:
lr = LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42)
lr.fit(X_train_scaled, y_train)

y_pred_lr = lr.predict(X_test_scaled)
y_prob_lr = lr.predict_proba(X_test_scaled)[:, 1]

print('=== Logistic Regression ===')
print(classification_report(y_test, y_pred_lr, target_names=['No', 'Yes']))
print(f'F1-Score (weighted): {f1_score(y_test, y_pred_lr, average="weighted"):.4f}')

### 5.2 Random Forest

In [ ]:
rf = RandomForestClassifier(n_estimators=200, class_weight='balanced', random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)

y_pred_rf = rf.predict(X_test)
y_prob_rf = rf.predict_proba(X_test)[:, 1]

print('=== Random Forest ===')
print(classification_report(y_test, y_pred_rf, target_names=['No', 'Yes']))
print(f'F1-Score (weighted): {f1_score(y_test, y_pred_rf, average="weighted"):.4f}')

### 5.3 Confusion Matrices

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for ax, y_pred, title in zip(
    axes,
    [y_pred_lr, y_pred_rf],
    ['Logistic Regression', 'Random Forest']
):
    cm = confusion_matrix(y_test, y_pred)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['No', 'Yes'])
    disp.plot(ax=ax, colorbar=False, cmap='Blues')
    ax.set_title(title, fontsize=12, fontweight='bold')

plt.suptitle('Confusion Matrices', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(CHARTS_DIR, 'confusion_matrices.png'), bbox_inches='tight')
plt.show()

### 5.4 ROC Curves

In [ ]:
plt.figure(figsize=(7, 5))

for y_prob, label, color in [
    (y_prob_lr, 'Logistic Regression', '#3498DB'),
    (y_prob_rf, 'Random Forest', '#E67E22')
]:
    fpr, tpr, _ = roc_curve(y_test, y_prob)
    roc_auc = auc(fpr, tpr)
    plt.plot(fpr, tpr, label=f'{label} (AUC = {roc_auc:.3f})', color=color, lw=2)

plt.plot([0, 1], [0, 1], 'k--', lw=1.5, label='Random Classifier')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curves — Model Comparison', fontsize=13, fontweight='bold')
plt.legend(loc='lower right')
plt.tight_layout()
plt.savefig(os.path.join(CHARTS_DIR, 'roc_curves.png'), bbox_inches='tight')
plt.show()

### 5.5 Feature Importance (Random Forest)

In [ ]:
importances = pd.Series(rf.feature_importances_, index=X.columns).sort_values(ascending=False)

plt.figure(figsize=(10, 6))
importances.head(15).plot(kind='bar', color='#3498DB', edgecolor='black')
plt.title('Top 15 Feature Importances — Random Forest', fontsize=13, fontweight='bold')
plt.ylabel('Importance Score')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig(os.path.join(CHARTS_DIR, 'feature_importance.png'), bbox_inches='tight')
plt.show()

## 6. Model Interpretability with SHAP

We use SHAP (SHapley Additive exPlanations) to explain **individual predictions** from the Random Forest model. SHAP values show how each feature contributes to pushing the prediction toward `Yes` or `No`.

In [ ]:
# Use TreeExplainer for Random Forest (fast & exact)
explainer = shap.TreeExplainer(rf)

# Compute SHAP values on the full test set
X_test_reset = X_test.reset_index(drop=True)
shap_values = explainer.shap_values(X_test_reset)

# shap_values[1] = SHAP values for class 1 (subscribed = Yes)
shap_vals_yes = shap_values[1]
print(f'SHAP values computed for {len(X_test_reset)} test samples.')

In [ ]:
# Global Summary Plot — which features matter most overall
plt.figure(figsize=(9, 7))
shap.summary_plot(shap_vals_yes, X_test_reset, show=False, max_display=15)
plt.title('SHAP Summary Plot — Random Forest (Class: Subscribed)', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(CHARTS_DIR, 'shap_summary.png'), bbox_inches='tight')
plt.show()

In [ ]:
# Select 5 diverse predictions to explain
# 3 correctly predicted positives + 2 correctly predicted negatives
y_pred_test = rf.predict(X_test_reset)
y_test_arr  = y_test.reset_index(drop=True)

tp_indices = X_test_reset[(y_pred_test == 1) & (y_test_arr == 1)].index[:3].tolist()
tn_indices = X_test_reset[(y_pred_test == 0) & (y_test_arr == 0)].index[:2].tolist()
explain_indices = tp_indices + tn_indices

print('Explaining predictions for sample indices:', explain_indices)

In [ ]:
# SHAP text summary for 5 individual predictions
for i, idx in enumerate(explain_indices):
    actual    = 'YES' if y_test_arr[idx] == 1 else 'NO'
    predicted = 'YES' if y_pred_test[idx] == 1 else 'NO'
    prob      = rf.predict_proba(X_test_reset.iloc[[idx]])[0, 1]

    print(f'\n--- Prediction {i+1} (Sample #{idx}) ---')
    print(f'  Actual: {actual} | Predicted: {predicted} | P(subscribe): {prob:.3f}')

    sample_shap = pd.Series(shap_vals_yes[idx], index=X_test_reset.columns)
    top5 = sample_shap.abs().sort_values(ascending=False).head(5)
    print('  Top 5 contributing features:')
    for feat in top5.index:
        direction = '+' if sample_shap[feat] > 0 else '-'
        print(f'    [{direction}] {feat}: value={X_test_reset.iloc[idx][feat]:.2f}, '
              f'SHAP={sample_shap[feat]:+.4f}')

In [ ]:
# Waterfall plots for each of the 5 predictions
fig, axes = plt.subplots(5, 1, figsize=(12, 22))

for i, (ax, idx) in enumerate(zip(axes, explain_indices)):
    actual    = 'YES' if y_test_arr[idx] == 1 else 'NO'
    predicted = 'YES' if y_pred_test[idx] == 1 else 'NO'
    prob      = rf.predict_proba(X_test_reset.iloc[[idx]])[0, 1]

    sample_shap = shap_vals_yes[idx]
    top_n = 8
    top_idx   = np.argsort(np.abs(sample_shap))[::-1][:top_n]
    top_feats = X_test_reset.columns[top_idx].tolist()
    top_shaps = sample_shap[top_idx]
    top_vals  = X_test_reset.iloc[idx, top_idx].values

    colors = ['#2ECC71' if v > 0 else '#E74C3C' for v in top_shaps]
    ax.barh(range(top_n), top_shaps, color=colors, edgecolor='black', alpha=0.8)
    ax.set_yticks(range(top_n))
    ax.set_yticklabels([f'{f} = {v:.2f}' for f, v in zip(top_feats, top_vals)], fontsize=8)
    ax.axvline(0, color='black', linewidth=0.8)
    ax.set_title(
        f'Prediction {i+1} | Actual: {actual} | Predicted: {predicted} | P(Yes)={prob:.3f}',
        fontsize=10, fontweight='bold'
    )
    ax.set_xlabel('SHAP value (impact on P(subscribe=Yes))')

plt.suptitle('SHAP Waterfall: Top Features for 5 Individual Predictions', fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig(os.path.join(CHARTS_DIR, 'shap_waterfall_5predictions.png'), bbox_inches='tight')
plt.show()

## 7. Model Comparison Summary

In [ ]:
fpr_lr, tpr_lr, _ = roc_curve(y_test, y_prob_lr)
fpr_rf, tpr_rf, _ = roc_curve(y_test, y_prob_rf)

summary = pd.DataFrame({
    'Model': ['Logistic Regression', 'Random Forest'],
    'F1-Score (weighted)': [
        round(f1_score(y_test, y_pred_lr, average='weighted'), 4),
        round(f1_score(y_test, y_pred_rf, average='weighted'), 4)
    ],
    'F1-Score (class=Yes)': [
        round(f1_score(y_test, y_pred_lr), 4),
        round(f1_score(y_test, y_pred_rf), 4)
    ],
    'ROC-AUC': [
        round(auc(fpr_lr, tpr_lr), 4),
        round(auc(fpr_rf, tpr_rf), 4)
    ]
})

print('=== Model Comparison ===')
summary

In [ ]:
# Print list of all saved charts
saved = sorted(os.listdir(CHARTS_DIR))
print(f'Charts saved to "{CHARTS_DIR}/" folder ({len(saved)} files):')
for f in saved:
    print(f'  - {f}')

## 8. Final Conclusion & Insights

### Key Findings

**1. Class Imbalance**  
The dataset is imbalanced (~88% No, ~12% Yes). We addressed this using `class_weight='balanced'` in both models to avoid biased predictions.

**2. Most Predictive Feature**  
`duration` (call duration in seconds) is by far the strongest predictor — longer calls strongly correlate with subscription. However, this feature is only known *after* a call ends, so it has limited use for pre-call targeting.

**3. Model Performance**  
- **Random Forest** outperforms Logistic Regression across all metrics (higher F1 and AUC)  
- Both models benefit from `class_weight='balanced'` to handle the class imbalance  
- ROC-AUC > 0.90 for Random Forest indicates strong discriminative ability

**4. Important Customer Signals**  
- Customers with a **successful previous campaign outcome** (`poutcome_success`) are far more likely to subscribe  
- Higher **account balance** and **previous contacts** also increase subscription probability  
- **Campaign contacts count** has a negative relationship — too many calls reduce likelihood

**5. SHAP Explanations (XAI)**  
SHAP analysis confirmed that `duration`, `poutcome_success`, `balance`, and `was_contacted_before` are the most impactful features. Individual waterfall plots showed how these features push predictions toward or away from subscription for specific customers.

### Business Recommendations
- Prioritize calling customers with **prior successful campaign history**
- Avoid over-contacting customers — diminishing returns after multiple calls
- Target customers with higher **account balances** for premium term deposit offers
- Use the Random Forest model for production scoring with a custom probability threshold (e.g., 0.3) to increase recall on the minority class